## Loading and Labelling the Data

In [1]:
import zipfile
import os

# List of files to unzip
zip_files = [
    'images_training_rev1.zip', 
    'images_test_rev1.zip', 
    'training_solutions_rev1.zip'
]

base_input_path = '/kaggle/input/competitions/galaxy-zoo-the-galaxy-challenge'
base_extract_path = '/kaggle/working/'

target_thresh = 0.8 #threshold for the data
RANDOM_SEED = 42

for file in zip_files:
    zip_path = os.path.join(base_input_path, file)
    
    # Create a specific folder for images, but keep the CSV in the main working dir
    if 'images' in file:
        folder_name = file.replace('.zip', '')
        extract_path = os.path.join(base_extract_path, folder_name)
    else:
        extract_path = base_extract_path
        
    os.makedirs(extract_path, exist_ok=True)

    print(f"Unzipping {file}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("All files extracted successfully.")

    location_path = os.path.join(base_extract_path, folder_name, folder_name)
    if os.path.exists(location_path):
        files = os.listdir(location_path)
        print(f"✅ Folder found!")
        print(f"Total files in training folder: {len(files)}")
        if len(files) > 0:
            print(f"Sample filename: {files[0]}")
    else:
        print("❌ Folder not found. Check your extraction path.")

Unzipping images_training_rev1.zip...
All files extracted successfully.
✅ Folder found!
Total files in training folder: 61578
Sample filename: 447267.jpg
Unzipping images_test_rev1.zip...
All files extracted successfully.
✅ Folder found!
Total files in training folder: 79975
Sample filename: 498477.jpg
Unzipping training_solutions_rev1.zip...
All files extracted successfully.
✅ Folder found!
Total files in training folder: 79975
Sample filename: 498477.jpg


In [2]:
#Hierarchical Thresholding

def thresh_labels(row, thresh):
    #the standards for the thresholds are 0.8 (from Willett et al. (2013), 0.5, and 0.7 (for the further tasks))
    # Scaled in this definition since lower quality images can be insightful testing data
    # Rule 1: High Consensus Smooth = Elliptical
    if row['Class1.1'] >= thresh:
        return 'Elliptical'
    
    # Rule 2: High Consensus Features + Arms = Spiral
    # (Note: We ignore the small % of 'Smooth' votes here)
    if row['Class1.2'] >= thresh and row['Class4.1'] >= ((5/8)*thresh):
        return 'Spiral'
    
    # Rule 3: High Consensus Artifact or 'Odd' = Irregular
    if row['Class1.3'] >= thresh or row['Class6.1'] >= ((7/8)*thresh):
        return 'Irregular'
    
    # Rule 4: If no consensus, throw it out! (Prevents noisy training)
    return 'Uncertain'

In [3]:
import pandas as pd

training_sol = pd.read_csv('/kaggle/working/training_solutions_rev1.csv')

# 3. Apply the function to create a new column
# 'args' passes the thresh value to your function
training_sol['label'] = training_sol.apply(thresh_labels, axis=1, thresh=target_thresh)

# 4. Filter out the 'Uncertain' rows to get your final labeled DF
train_sol_labeled = training_sol[training_sol['label'] != 'Uncertain'].copy()

# 5. Check the results
print(f"Original size: {len(training_sol)}")
print(f"Labeled size: {len(train_sol_labeled)}")
print(train_sol_labeled['label'].value_counts())

Original size: 61578
Labeled size: 20529
label
Spiral        9107
Elliptical    8132
Irregular     3290
Name: count, dtype: int64


## The Stratified 80/10/10 Split

In [4]:
from sklearn.model_selection import train_test_split

# Create the 80/10/10 split
# 1. First split: 80% Train, 20% "Remainder"
train_df, temp_df = train_test_split(
    train_sol_labeled, 
    test_size=0.2, 
    stratify=train_sol_labeled['label'], 
    random_state=RANDOM_SEED
)

# 2. Second split: Split the 20% "Remainder" into half (10% Val, 10% Test)
# FIX: Use temp_df here, not the full train_sol_labeled
val_df, test_df = train_test_split(
    temp_df, 
    test_size=0.5, 
    stratify=temp_df['label'], 
    random_state=RANDOM_SEED
)